# Industrial Graded RAG implementation experiment

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

True

# Setup Confituration

In [ ]:
class Config:
    # setup mistral configuration
    mistral_api_key = os.getenv("MISTRAL_API_KEY")
    mistral_chat_model = os.getenv("MISTRAL_CHAT_MODEL")
    mistral_embed_model = "mistral-embed"
    mistral_embed_dimension =  int(os.getenv("MISTRAL_EMBED_DIMENSION")) or 1024

    # pinecone configuration
    pinecone_api_key = os.getenv("PINECONE_API_KEY")
    pinecone_index = "rag-system"
    pinecone_namespace = os.getenv("PINECONE_NAMESPACE")
    pinecone_region = os.getenv("PINECONE_REGION")

    # document path details
    document_dir = "docs"

    # text splitter
    chunk_size = 1000
    overlap_size = 200

# LLM Service setup

In [3]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

class MistralService:
    chatModel : ChatMistralAI
    embeddingModel : MistralAIEmbeddings

    def __init__(self):
        self.chatModel = self.connectMistralChatModel()
        self.embeddingModel = self.connectMistralEmbedModel()

    def connectMistralChatModel(self, model_name : str = Config.mistral_chat_model) -> ChatMistralAI :
        try:
            return ChatMistralAI(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
        
    def connectMistralEmbedModel(self, model_name: str = Config.mistral_embed_model) -> MistralAIEmbeddings :
        try:
            return MistralAIEmbeddings(
                api_key = Config.mistral_api_key,
                model = model_name
            )
        except Exception as e:
            print(f"Something went wrong in the {model_name} connection...")
            raise
    
    def getChatModel(self) -> ChatMistralAI:
        return self.chatModel
    
    def getEmbedModel(self) -> MistralAIEmbeddings:
        return self.embeddingModel

# Pinecone Setup

In [4]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

class PineconeService:
    def __init__(self):
        self.pc = Pinecone(api_key=Config.pinecone_api_key) 
        self.prepare_pinecone()

    def prepare_pinecone(self):
        index_list_obj = self.pc.list_indexes()
        index_names = [index.name for index in index_list_obj]

        spec = ServerlessSpec(
            cloud="aws",
            region=Config.pinecone_region
        )

        if Config.pinecone_index not in index_names:
            try:
                print(f"Creating index: {Config.pinecone_index}")
                self.pc.create_index(
                    name=Config.pinecone_index,
                    dimension=Config.mistral_embed_dimension,
                    metric="cosine",
                    spec=spec
                )
            except Exception as e:
                print(f"Error during index creation: {e}")
                raise
        else:
            print(f"Index '{Config.pinecone_index}' already exists. Skipping creation.")

    def getPinecone(self) -> Pinecone:
        return self.pc

    def getPineconeStore(self, embedding) -> PineconeVectorStore:
        self.vectorStore = PineconeVectorStore(
            index_name= Config.pinecone_index,
            embedding= embedding
        )

        return self.vectorStore

/home/buddhika-madusanka/projects/industrial-rag-system/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Document Chunker

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

class TextSplitter:
    splitter: RecursiveCharacterTextSplitter

    def __init__(self):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size = Config.chunk_size,
            chunk_overlap = Config.overlap_size,
            length_function =len 
        ) 
    
    def getSplitter(self):
        return self.splitter

# Document Reader

In [6]:
from langchain_core.documents import Document
from datetime import datetime

class DocumentPrepare:

    def __init__(self, document_name, document_cotent) :
        self.document = Document(
            page_content= document_cotent,
            metadata = {
                "document_name" : document_name,
                "document_created_at" :  datetime.now().isoformat()
            }
        )
    
    def getDocument(self):
        return self.document

In [7]:
from PyPDF2 import PdfReader

class PdfDocumentCotentReader:
    reader: PdfReader

    def __init__(self, path:str):
        self.reader = PdfReader(path)

    def cleanText(self, content: str) -> str:
        return content.replace("/n", " ").replace("/t", " ").strip()
    
    def scrapeContent(self) -> str:
        page_count = len(self.reader.pages)
        content = ""

        for i in range(page_count):
            page = self.reader.pages[i]
            page_content = page.extract_text()
            content += " " + page_content

        return content
    
    def getContent(self):
        raw_content = self.scrapeContent()
        content = self.cleanText(raw_content)
        return content

In [9]:
from pathlib import Path

def getExtension(file_name: str) -> str:
    return file_name.split(".")[-1]

class DocumentContentExtractor:
    content: str

    def __init__(self):
        pass

    def contentExtract(self):
        document_path = Config.document_dir
        path_obj = Path(document_path)

        if path_obj.exists() and path_obj.is_dir():
            files = [f for f in path_obj.iterdir() if f.is_file()]
            base_url = Path(document_path)

        for file in files:
            print(f"=" * 50)
            print(file.name)
            print(f"=" * 50)
            file_path = base_url / file.name
            file_extensiton = getExtension(file.name)

            if file_extensiton.lower() == "pdf":
                reader = PdfDocumentCotentReader(file_path)
                self.content = reader.getContent()

            document_obj = DocumentPrepare(
                document_name= file.name,
                document_cotent= self.content
            )
            document = document_obj.getDocument()

            splitter = TextSplitter().getSplitter()
            docs = splitter.split_documents([document])

            pc = PineconeService()
            vectorStore = pc.getPineconeStore(
                MistralService().getEmbedModel()
            )

            vectorStore.add_documents(docs)
            print(docs)
            
        return self.content
    
content_obj = DocumentContentExtractor()
content = content_obj.contentExtract()

ai.pdf
Index 'rag-system' already exists. Skipping creation.


An error occurred with MistralAI
Traceback (most recent call last):
  File "/home/buddhika-madusanka/projects/industrial-rag-system/.venv/lib/python3.13/site-packages/tenacity/__init__.py", line 473, in __call__
    result = fn(*args, **kwargs)
  File "/home/buddhika-madusanka/projects/industrial-rag-system/.venv/lib/python3.13/site-packages/langchain_mistralai/embeddings.py", line 257, in _embed_batch
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/home/buddhika-madusanka/projects/industrial-rag-system/.venv/lib/python3.13/site-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '400 Bad Request' for url 'https://api.mistral.ai/v1/embeddings'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/buddh

RetryError: RetryError[<Future at 0x7fc920430510 state=finished raised HTTPStatusError>]